# 🧪 Lab 04 — ClosureCleaner + Spark Serialization Diagnostics with MadLava Scopes

This notebook compares the same deterministic Spark ML workload under:

1. Java serialization
2. Kryo serialization
3. Kryo with explicit class registration

MadLava owns the profiling interval and presentation. Each experiment is delimited
by a named MadLava scope:

```text
create SparkContext
    ↓
beginScope(mode)
    ↓
workload
    ↓
endScope(scope)
    ↓
ScopeResult
    ↓
MadLavaReport.scopeReportText(result)
    ↓
stop SparkContext
```

The notebook deliberately does **not**:

- create or release MadLava checkpoints;
- parse Runtime Statistics JSON;
- aggregate profiler counters;
- build ClosureCleaner tables;
- build Java/Kryo serializer tables;
- calculate report summaries.

All of that belongs to MadLava Iteration-12.

Each serializer phase produces **its own independent MadLava scope report**.
The notebook intentionally does not request or print a cumulative whole-JVM report.

Expected report semantics for every phase:

```text
Trigger         : SCOPE_END
Statistics Mode : SCOPE_DELTA
Scope           : java | kryo | registered_kryo
```


## 1 — Configuration and prerequisites

Keep these files beside the notebook:

```text
madlava-agent-0.1.0.jar
madlava.json
```

The lab requires:

```json
{
  "features": {
    "methodProfiling": {
      "enabled": true
    },
    "sparkSerialization": {
      "enabled": true,
      "profile": "ALL"
    }
  },
  "filters": {
    "methods": {
      "includes": [
        "org.apache.spark.util.ClosureCleaner$.clean"
      ]
    }
  }
}
```

No `reporting.output` is required for the interactive notebook. Automatic human
report persistence may be enabled independently in MadLava configuration, but the
notebook only consumes the in-process reporting API.


In [ ]:
import os
import time

# ----------------------------------------------------------------------
# MadLava JVM bootstrap
# ----------------------------------------------------------------------
# The Java agent must be present on the command that launches PySpark's
# gateway JVM. The shared JSON is therefore attached through
# PYSPARK_SUBMIT_ARGS before any SparkContext/SparkSession is created.

MADLAVA_JAR = os.path.abspath("madlava-agent-0.1.0.jar").replace("\\", "/")
MADLAVA_CONFIG = os.path.abspath("madlava.json").replace("\\", "/")
MADLAVA_REPORTS = {}

for _path in (MADLAVA_JAR, MADLAVA_CONFIG):
    if not os.path.isfile(_path):
        raise FileNotFoundError(f"Missing required MadLava file: {_path}")

_MADLAVA_AGENT_OPTION = (
    f"-javaagent:{MADLAVA_JAR}=config={MADLAVA_CONFIG}"
)

_existing_submit_args = os.environ.get("PYSPARK_SUBMIT_ARGS", "").strip()

# Remove the shell marker so our --conf is inserted before it.
if _existing_submit_args.endswith("pyspark-shell"):
    _existing_submit_args = _existing_submit_args[:-len("pyspark-shell")].strip()

if "--driver-java-options" in _existing_submit_args:
    raise RuntimeError(
        "PYSPARK_SUBMIT_ARGS already defines --driver-java-options. "
        "Restart the kernel after removing that conflicting definition."
    )

os.environ["PYSPARK_SUBMIT_ARGS"] = (
    f'{_existing_submit_args} '
    f'--driver-java-options "{_MADLAVA_AGENT_OPTION}" '
    f'pyspark-shell'
).strip()

from pyspark import SparkContext
from pyspark.sql import SparkSession
from pyspark.ml.linalg import Vectors
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.evaluation import BinaryClassificationEvaluator

MODES = ["java", "kryo", "registered_kryo"]

print("✅ MadLava launch configuration prepared")
print(f"   Agent : {MADLAVA_JAR}")
print(f"   Config: {MADLAVA_CONFIG}")


## 2 — Single-JVM Spark lifecycle

The agent can only be attached when the PySpark gateway JVM starts. If this kernel
already owns a gateway JVM without MadLava, restart the kernel once and run the
notebook from the top.

Only Spark contexts are recreated between phases; the Java gateway and MadLava
runtime remain alive.


In [ ]:
def _gateway_madlava_available() -> bool:
    if SparkContext._gateway is None:
        return False
    try:
        api = SparkContext._gateway.jvm.com.madlava.api.MadLavaStatistics
        return bool(api.isAvailable())
    except Exception:
        return False


_EXISTING_GATEWAY = SparkContext._gateway is not None
_AGENT_BOOTSTRAPPED = _gateway_madlava_available()
_DRIVER_PID = None
_CONTEXT_INDEX = 0

if _EXISTING_GATEWAY and not _AGENT_BOOTSTRAPPED:
    raise RuntimeError(
        "A PySpark gateway JVM already exists but MadLava is not attached. "
        "Restart the kernel once, then Run All."
    )


def _driver_pid(spark: SparkSession) -> int:
    return int(
        spark.sparkContext._jvm.java.lang.ProcessHandle.current().pid()
    )


def stop_spark_keep_jvm(spark: SparkSession):
    if spark is None:
        return
    pid = _driver_pid(spark)
    print(f"⚰️ SparkContext stopped; MadLava JVM PID {pid} remains alive")
    spark.stop()
    time.sleep(0.75)


if _AGENT_BOOTSTRAPPED and SparkContext._active_spark_context is not None:
    active = SparkContext._active_spark_context
    pid = int(active._jvm.java.lang.ProcessHandle.current().pid())
    print(f"♻️ Stopping pre-existing SparkContext in MadLava JVM PID {pid}")
    active.stop()
    time.sleep(0.75)


def build_diagnostic_spark(mode: str) -> SparkSession:
    global _AGENT_BOOTSTRAPPED, _DRIVER_PID, _CONTEXT_INDEX

    if mode not in MODES:
        raise ValueError(f"Unsupported mode: {mode}")
    if SparkContext._active_spark_context is not None:
        raise RuntimeError("A SparkContext is still active.")

    launching_gateway = not _AGENT_BOOTSTRAPPED

    builder = (
        SparkSession.builder
        .master("local[*]")
        .appName(f"madlava-closure-diagnostics-{mode}")
        .config("spark.driver.memory", "4g")
        .config("spark.sql.shuffle.partitions", "8")
        .config("spark.ui.enabled", "false")
    )


    if mode == "java":
        builder = builder.config(
            "spark.serializer",
            "org.apache.spark.serializer.JavaSerializer",
        )
    else:
        builder = builder.config(
            "spark.serializer",
            "org.apache.spark.serializer.KryoSerializer",
        )

    if mode == "registered_kryo":
        builder = (
            builder
            .config("spark.kryo.registrationRequired", "false")
            .config(
                "spark.kryo.classesToRegister",
                ",".join([
                    "org.apache.spark.ml.linalg.DenseVector",
                    "org.apache.spark.ml.linalg.SparseVector",
                    "org.apache.spark.ml.classification.LogisticRegressionModel",
                ]),
            )
        )

    spark = builder.getOrCreate()
    spark.sparkContext.setLogLevel("ERROR")

    if not _gateway_madlava_available():
        runtime_args = [
            str(arg)
            for arg in spark.sparkContext._jvm.java.lang.management
                .ManagementFactory.getRuntimeMXBean()
                .getInputArguments()
        ]
        javaagent_args = [
            arg for arg in runtime_args if arg.startswith("-javaagent:")
        ]
        raise RuntimeError(
            "MadLava did not activate in the PySpark gateway JVM. "
            f"Observed JVM -javaagent arguments: {javaagent_args or ['<none>']}. "
            "If the MadLava argument is present, inspect JVM startup stderr for "
            "'bootstrap disabled'; that indicates the agent rejected its startup "
            "configuration. Restart the kernel after correcting the cause."
        )

    pid = _driver_pid(spark)
    if _DRIVER_PID is None:
        _DRIVER_PID = pid
    elif pid != _DRIVER_PID:
        raise RuntimeError(
            f"Expected one persistent JVM, but PID changed from {_DRIVER_PID} to {pid}."
        )

    _AGENT_BOOTSTRAPPED = True
    _CONTEXT_INDEX += 1

    label = "started" if launching_gateway else "reused"
    print(f"🔥 MadLava JVM {label} — PID {pid}")
    return spark


## 3 — MadLava scope + native report client

Scopes are the high-level API for named profiling intervals. Checkpoints remain
an internal/advanced MadLava primitive and are not exposed in this notebook.

The public flow used here is:

```text
MadLavaScopes.beginScope(name)
        ↓
workload
        ↓
MadLavaScopes.endScope(scopeId)
        ↓
scopeResultId
        ↓
MadLavaReport.scopeReportText(scopeResultId)
```

The native report should identify itself as:

```text
Trigger         : SCOPE_END
Statistics Mode : SCOPE_DELTA
Scope           : <phase name>
```

No Python-side report reconstruction or reflection-based `Class.forName()` checks
are used.


In [ ]:
class MadLavaScopeReports:
    """
    Thin Py4J adapter over the public Iteration-12 scope/report APIs.

    Python intentionally knows nothing about the profiler statistics schema.
    """

    def __init__(self, spark: SparkSession):
        self.jvm = spark.sparkContext._jvm
        self.statistics = self.jvm.com.madlava.api.MadLavaStatistics
        self.scopes = self.jvm.com.madlava.api.MadLavaScopes
        self.reports = self.jvm.com.madlava.api.MadLavaReport

        if not bool(self.statistics.isAvailable()):
            raise RuntimeError(
                "MadLavaStatistics.isAvailable() returned false. "
                "Restart the notebook kernel and make sure the Iteration-12 "
                "MadLava agent JAR is attached to the PySpark gateway JVM."
            )

        try:
            scopes_available = bool(self.scopes.isAvailable())
        except Exception as exc:
            raise RuntimeError(
                "The running MadLava JAR does not expose the Iteration-12 "
                "MadLavaScopes API. Rebuild madlava-agent-0.1.0.jar from the "
                "updated Iteration-12 branch, restart the notebook kernel, "
                "and Run All."
            ) from exc

        if not scopes_available:
            raise RuntimeError("MadLavaScopes.isAvailable() returned false.")

    def begin_scope(self, name: str) -> str:
        try:
            scope_id = str(self.scopes.beginScope(name))
        except Exception as exc:
            raise RuntimeError(
                f"MadLava could not begin scope {name!r}."
            ) from exc

        if not scope_id:
            raise RuntimeError(
                f"MadLava returned an empty scope ID for {name!r}."
            )

        return scope_id

    def end_scope(self, scope_id: str) -> str:
        try:
            result_id = str(self.scopes.endScope(scope_id))
        except Exception as exc:
            raise RuntimeError(
                f"MadLava could not end scope {scope_id!r}."
            ) from exc

        if not result_id:
            raise RuntimeError(
                f"MadLava returned an empty ScopeResult ID for {scope_id!r}."
            )

        return result_id

    def scope_report_text(self, result_id: str) -> str:
        try:
            report = str(self.reports.scopeReportText(result_id))
        except Exception as exc:
            raise RuntimeError(
                "MadLava completed the scope, but "
                f"MadLavaReport.scopeReportText({result_id!r}) failed."
            ) from exc

        if not report.strip():
            raise RuntimeError(
                f"MadLava returned an empty native report for {result_id!r}."
            )

        return report


## 4 — Deterministic Spark ML workload

The workload is identical for all serializer modes.

The `SparkContext` is created **before** `beginScope()` and stopped **after**
`endScope()`. Therefore each MadLava `SCOPE_DELTA` contains the experiment itself,
not SparkContext startup or teardown.

MadLava measures the official scope duration. The Python timer remains only as a
small workload sanity check and is not used for profiler reporting.


In [ ]:
def generate_micro_dataset(spark: SparkSession):
    rows = []
    for i in range(80):
        label = 1.0 if i % 2 == 0 else 0.0
        features = Vectors.dense([
            float(i * 0.1),
            float((i % 3) * 0.5),
            float(label * 0.2),
        ])
        rows.append((label, features))

    return spark.createDataFrame(
        rows, ["label", "features"]
    ).repartition(4)


def execute_closure_workload(spark: SparkSession) -> dict:
    df = generate_micro_dataset(spark)

    lr = LogisticRegression(
        maxIter=10,
        featuresCol="features",
        labelCol="label",
    )

    evaluator = BinaryClassificationEvaluator(
        rawPredictionCol="rawPrediction",
        labelCol="label",
    )

    grid = (
        ParamGridBuilder()
        .addGrid(lr.regParam, [0.1, 0.01, 0.001])
        .addGrid(lr.elasticNetParam, [0.0, 0.5])
        .build()
    )

    cv = CrossValidator(
        estimator=lr,
        estimatorParamMaps=grid,
        evaluator=evaluator,
        numFolds=5,
        parallelism=1,
    )

    started = time.perf_counter()
    model = cv.fit(df)
    prediction_count = model.transform(df).count()

    return {
        "duration_seconds": time.perf_counter() - started,
        "prediction_count": prediction_count,
    }


## 5 — Execute Java → Kryo → registered Kryo

Each phase is now a first-class MadLava scope:

```text
JAVA             → scope "java"
KRYO             → scope "kryo"
REGISTERED_KRYO  → scope "registered_kryo"
```

The notebook prints the native scope report verbatim. It does not query checkpoints,
parse statistics JSON, or construct any profiler table itself.


In [ ]:
phase_results = {}

for mode in MODES:
    print("\n" + "=" * 110)
    print(f"🔥 STARTING LAB PHASE [{mode.upper()}]")
    print("=" * 110)

    spark = build_diagnostic_spark(mode)
    madlava = MadLavaScopeReports(spark)
    scope_id = madlava.begin_scope(mode)
    print(f"📍 MadLava scope started: {scope_id} [{mode}]")

    workload = None
    result_id = None
    native_report = None

    try:
        workload = execute_closure_workload(spark)

        # endScope() materializes one immutable ScopeResult from MadLava's
        # internal checkpoint/delta machinery.
        result_id = madlava.end_scope(scope_id)

        # Presentation is entirely owned by MadLava.
        native_report = madlava.scope_report_text(result_id)

    except Exception:
        # Do not silently call endScope() a second time. Scope closure is a
        # single-shot semantic operation. Preserve the original failure.
        raise

    phase_results[mode] = {
        "scope_id": scope_id,
        "result_id": result_id,
        "driver_pid": _driver_pid(spark),
        "spark_version": spark.version,
        "serializer": spark.conf.get("spark.serializer"),
        "report": native_report,
        **workload,
    }

    print(
        f"✅ Workload completed: {workload['prediction_count']} rows, "
        f"{workload['duration_seconds']:.2f} s"
    )
    print(f"📦 MadLava ScopeResult: {result_id}")
    print(f"📄 MadLava native scope report [{mode}]\n")

    # IMPORTANT: render exactly what MadLava produced.
    print(native_report)

    stop_spark_keep_jvm(spark)

pids = {result["driver_pid"] for result in phase_results.values()}
if len(pids) != 1:
    raise RuntimeError(
        f"Expected one persistent MadLava JVM, observed PIDs: {sorted(pids)}"
    )

print(
    f"\n✅ All three MadLava scopes completed in JVM PID {_DRIVER_PID}."
)


## 6 — Measurement boundaries

For each phase:

```text
SparkContext creation        excluded
MadLava beginScope           scope starts
ML/CrossValidator workload   included
MadLava endScope             scope ends + immutable ScopeResult
MadLava scopeReportText      presentation only
SparkContext.stop            excluded
```

The notebook is deliberately thin.

MadLava owns:

- internal checkpoint/baseline management;
- scope delta semantics;
- canonical argument grouping;
- method profiling aggregation;
- Spark serializer summary/detail aggregation;
- diagnostics and consistency warnings;
- report trigger/statistics-mode metadata;
- sorting and truncation;
- Spark-show-style table rendering;
- optional scope-end and JVM-shutdown file persistence.

Python owns only the experiment workload and Spark lifecycle.


# 📊 Post-Lab Analysis: The Closure Inspection That Keeps Returning

This diagnostics lab puts hard profiler evidence behind the central ClosureCleaner question: **does switching Spark from Java serialization to Kryo actually make Spark clean fewer closures?**

The answer from this run is remarkably clear. The dataset contains only **80 rows**, but the Spark ML orchestration surrounding those rows repeatedly constructs, cleans, validates, serializes, and executes distributed functions. MadLava lets us separate that control-plane activity from the serializer implementation selected for the application data path.

---

## 1. The 5,270-Call Funeral Inspection

Across all three serializer configurations, MadLava recorded exactly the same number of calls to:

`org.apache.spark.util.ClosureCleaner$.clean`

* **JavaSerializer:** 5,270 invocations
* **KryoSerializer:** 5,270 invocations
* **Registered KryoSerializer:** 5,270 invocations

That is the strongest result in the entire experiment.

Changing `spark.serializer` changes the serializer Spark can use for applicable JVM data paths, but it does **not** change how many times this Spark ML workload asks `ClosureCleaner` to inspect a closure before distributed execution.

The body is microscopic. The paperwork is not.

Five folds, six parameter combinations, iterative logistic-regression optimization, evaluation, RDD transformations, and final-model execution repeatedly manufacture distributed-function boundaries. `ClosureCleaner` is responding to that orchestration graph, not to the physical number of rows in the DataFrame.

---

## 2. RDDLossFunction: The Repeat Offender

The argument-level method telemetry identifies Spark ML's iterative optimizer as one of the largest repeat visitors to the inspection desk.

In the Java scope alone, the two dominant `RDDLossFunction` lambda families account for:

* **1,725** calls from `RDDLossFunction$$Lambda$4995`
* **345** calls from `RDDLossFunction$$Lambda$4994`

That is **2,070 ClosureCleaner invocations** attributable to only those two optimizer lambda families—roughly **39% of all 5,270 `clean()` calls** in the phase.

This is why a dataset containing only 80 rows can still generate thousands of closure-cleaning operations. Iterative machine-learning orchestration repeatedly creates distributed loss-evaluation work. The optimizer keeps coming back to the funeral desk with another form to stamp.

The important variable is therefore not merely:

`number of rows`

but:

`number of distributed function boundaries created by the orchestration`

---

## 3. Kryo Changes the Hearse, Not the Number of Funerals

The Spark serialization report makes the separation visible.

Under the Java baseline, MadLava observes the Java serializer family throughout the workload.

Once `spark.serializer` is switched to Kryo, the report begins recording Kryo activity such as:

* `KryoSerializer`
* `KryoSerializerInstance`
* `KryoSerializationStream`
* `KryoDeserializationStream`

But Java serialization does **not** disappear.

The Kryo scopes still contain substantial activity from:

* `JavaSerializer`
* `JavaSerializerInstance`
* `JavaSerializationStream`

This is the architectural point the lab was designed to expose: selecting Kryo as the application serializer does not replace every Java-serialization path inside Spark.

Spark still has internal duties—most importantly closure/task preparation paths—that remain structurally separate from the configurable serializer used for applicable data payloads.

So Kryo can change **how some objects are serialized**.

It does not magically change **how often Spark decides that distributed work must be prepared, cleaned, and serialized**.

---

## 4. Registered Kryo Does Not Reduce Serialization Frequency

The standard Kryo and registered Kryo phases produce the same ClosureCleaner invocation count:

**5,270 vs. 5,270**

Their Spark serialization call patterns are also structurally almost identical.

That is exactly what we should expect.

Class registration optimizes Kryo's representation of known classes. Instead of repeatedly carrying verbose class metadata, Kryo can identify registered types using compact internal IDs.

Registration can therefore affect:

* encoded payload size;
* class lookup work;
* serializer CPU cost;
* allocation pressure.

It does **not** remove the Spark operations that caused serialization to happen in the first place.

Registration changes the luggage tags.

It does not cancel the flight.

---

## 5. Count and Cost Are Different Crimes

MadLava also measured the cumulative time spent inside `ClosureCleaner.clean`:

* **JavaSerializer:** 10.84 seconds
* **KryoSerializer:** 5.61 seconds
* **Registered KryoSerializer:** 8.74 seconds

The corresponding end-to-end workload times were:

* **JavaSerializer:** 313.44 seconds
* **KryoSerializer:** 253.62 seconds
* **Registered KryoSerializer:** 285.35 seconds

These timing differences are interesting, but they are **not sufficient to claim that Kryo makes ClosureCleaner faster**.

All three phases intentionally execute inside the same persistent JVM so that the same MadLava agent can own all three scopes. That means later phases inherit a warmer JVM: loaded classes, JIT-compiled code, populated Spark internals, filesystem caches, and other runtime state can all influence elapsed time.

The invocation count is therefore the cleaner structural result.

The timing numbers tell us something narrower: repeated closure cleaning is not theoretical bookkeeping. In this run it accumulated **seconds of measurable JVM work**. But this notebook is a diagnostic experiment, not a controlled serializer speed benchmark.

A crowded corridor is measurable.

That does not automatically make it the building's only fire.

---

## 6. The Dual-Serializer Architecture Is No Longer an Abstraction

The profiler output gives us a direct view of Spark operating with two serializer families at the same time.

In the Kryo phases, MadLava simultaneously observes Java and Kryo serializer operations.

That matters because `spark.serializer` is often mentally treated as a global replacement switch:

```text
Java OFF
Kryo ON
```

The runtime evidence shows a much more accurate picture:

```text
                         Spark JVM
                            │
             ┌──────────────┴──────────────┐
             │                             │
      Closure / task paths          Configurable data paths
             │                             │
             ▼                             ▼
      Java serialization        Java or Kryo via spark.serializer
```

This explains the apparent paradox at the heart of the experiment.

You can enable Kryo correctly.

You can register classes correctly.

And Spark can still execute thousands of Java-serialization operations because those operations belong to a different responsibility inside the engine.

Kryo did not fail.

It was simply never appointed undertaker for every funeral in the building.

---

## 7. What the Lab Actually Proves

The experiment supports five concrete conclusions:

1. **ClosureCleaner activity is driven by orchestration, not dataset size alone.**  
   An 80-row dataset generated 5,270 `ClosureCleaner.clean` invocations per phase.

2. **The serializer choice does not reduce the number of closure-cleaning decisions.**  
   Java, Kryo, and registered Kryo all produced exactly 5,270 calls.

3. **Spark ML iterative optimization is a major source of repeated closure preparation.**  
   `RDDLossFunction` lambda families dominate a substantial portion of the observed calls.

4. **Java serialization remains active when `spark.serializer` is Kryo.**  
   MadLava directly observes both Java and Kryo serializer families in the Kryo scopes.

5. **Class registration optimizes representation, not orchestration frequency.**  
   Registered Kryo changes how known types can be encoded; it does not eliminate the Spark operations that invoke serialization.

---

> ### ⚰️ Forensic Verdict
> The murder weapon was never "Java serialization" alone. The deeper tax is the number of times Spark's orchestration layer walks back to the same inspection desk. Kryo can give the payload a faster hearse, and class registration can strip the luggage from the coffin, but neither one stops `ClosureCleaner` from checking the paperwork **5,270 times**. If repeated closure preparation becomes a meaningful bottleneck, the remedy lives in reducing unnecessary orchestration and repeated distributed-function construction—not in expecting a different serializer to make those decisions disappear.
